Loaded all data, seperated truth from input data.
Standardised input data for staple results with NN
Plottet correlation Mmatrix to survey for correlated features to be excluded.
Ran mutual info classify from scikit learn to get an overview of which features reduced the uncertanty of y the most.
removed correlated features where the correlation was greater than .95, keeping the feature in each pair with the highest MI score
Choose the top 15 features as input features for NN.

This apporach was chosen as an alternative to just doing correlation removal and then looking at feature importance of a model trained over all features due to the vast computationalæ needs to repeatedly train a nn with 100 inputfeatures.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif


# Load the dataset
data = pd.read_hdf('/Users/prometheus/Documents/Python/AppliedML2026/InitialProject/IP_main/AppML_InitialProject_train.h5')
y = data['p_Truth_isElectron']
data.drop(columns=['p_Truth_isElectron'], inplace=True)
X = data
#see if any nans or infs in the data or if any features have zero variance and dtype of each feature
#print(X.isna().sum())

#standardise the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

#X.head()



In [2]:
# correlation matrix
#plt.figure(figsize=(12, 10))
#correlation_matrix = X.corr()
#sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
#plt.title('Correlation Matrix of Features')
#plt.show()

In [3]:


mi = mutual_info_classif(X, y)

# Run MI first
mi_scores = pd.Series(mutual_info_classif(X, y), index=X.columns)

# Then drop the lower-MI feature from each correlated pair
upper = X.corr().abs()
upper = upper.where(np.triu(np.ones(upper.shape), k=1).astype(bool))

to_drop = set()
for col in upper.columns:
    correlated_with = upper.index[upper[col] > 0.95].tolist()
    for other in correlated_with:
        # Drop whichever has lower MI
        if mi_scores[col] < mi_scores[other]:
            to_drop.add(col)
        else:
            to_drop.add(other)

X = X.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} highly correlated features")


Dropped 44 highly correlated features


In [4]:
# Update MI scores for remaining features

mi = mutual_info_classif(X, y)

mi_scores = pd.Series(mutual_info_classif(X, y), index=X.columns)

mi_scores.sort_values(ascending=False, inplace=True)



In [5]:
# Assigns the 15 highest MI scores to X to use as input features for the model
X = X[mi_scores.head(15).index]

#X.head(15)



In [6]:
#save list of features to use in the model to features.txt
with open('NN_input_features.txt', 'w') as f:
    for feature in X.columns:
        f.write(f"{feature}\n")